In [ ]:
import os
import tensorflow as tf
import cv2
import numpy as np
import gradio as gr
import time
import random

if "TF_USE_LEGACY_KERAS" in os.environ:
    del os.environ["TF_USE_LEGACY_KERAS"]

class CompatibleBatchNormalization(tf.keras.layers.BatchNormalization):
    def __init__(self, **kwargs):
        # Bỏ các tham số thừa của Keras 2 khiến Keras 3 bị lỗi
        kwargs.pop('renorm', None)
        kwargs.pop('renorm_clipping', None)
        kwargs.pop('renorm_momentum', None)
        super().__init__(**kwargs)

FILE_ID = "11VFv9agOvWlr4oUNHyfRcREqNIgvEvBD"   # ID file của bạn
MODEL_PATH = "zoo_efficientnetb0_colab.keras"                   # Tên file lưu mô hình (model mới lưu dạng .keras)

# Tải file mô hình (Nếu chưa có)
if not os.path.exists(MODEL_PATH):
    !pip install -q gdown
    !gdown {FILE_ID} -O {MODEL_PATH}

# LOAD MODEL: Sử dụng Keras 3 gốc kết hợp với lớp sửa lỗi tự chế ở trên
model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={'BatchNormalization': CompatibleBatchNormalization}
)

IMG_WIDTH = 256
IMG_HEIGHT = 256

CLASS_NAMES = [
    'chicken',    # 0
    'dog',        # 1
    'spider',     # 2
    'squirrel',   # 3
    'capybara',   # 4
    'frog',       # 5
    'rhino',      # 6
    'turtle',     # 7
    'tiger',      # 8
    'crocodile',  # 9
    'wolf',       # 10
    'horse',      # 11
    'elephant',   # 12
    'giraffe',    # 13
    'deer',       # 14
    'goat',       # 15
    'butterfly',  # 16
]

# Thứ tự output của model = thứ tự thư mục mà image_dataset_from_directory sắp theo CHỮ
# (giống class_names.json): output 2 là thư mục "10" (wolf), KHÔNG phải "2" (spider).
# Phải đổi idx của model -> số thư mục trước khi tra CLASS_NAMES.
MODEL_CLASS_ORDER = ["0", "1", "10", "11", "12", "13", "14", "15", "16",
                     "2", "3", "4", "5", "6", "7", "8", "9"]

CONFIDENCE_THRESHOLD = 0.65

# HÀM DỰ ĐOÁN

def predict_animal(image):
    img = np.asarray(image)

    if img.ndim == 2:                          # ảnh xám -> 3 kênh
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[-1] == 4:                   # PNG có alpha -> bỏ alpha
        img = img[..., :3]

    img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
    img = img.astype(np.uint8)
    batch = np.expand_dims(img, axis=0)

    probs = model.predict(batch, verbose=0)[0]
    idx = int(np.argmax(probs))
    return CLASS_NAMES[int(MODEL_CLASS_ORDER[idx])], float(probs[idx])

#==============================================================
# DỮ LIỆU LOÀI
#==============================================================
animal_info = {
    'chicken': {
        'lifespan': '5-8 years in the wild (up to 15 years in captivity/zoos).',
        'habitat': "Originally native to the tropical forests of Southeast Asia (Red Junglefowl). Now found globally as domesticated livestock, thriving in farms and backyard coops.",
        'wiki': 'https://en.wikipedia.org/wiki/Chicken',
    },
    'dog': {
        'lifespan': '4-8 years in the wild (up to 13 years in captivity/zoos).',
        'habitat': "Distributed worldwide across every continent except Antarctica. They live in close association with humans in urban and rural areas; some feral packs roam scrublands or deserts.",
        'wiki': 'https://en.wikipedia.org/wiki/Dog',
    },
    'spider': {
        'lifespan': '1-2 years in the wild (up to 20 years in captivity/zoos).',
        'habitat': "Found on every continent except Antarctica. They are highly adaptable, thriving in nearly every habitat including rainforests, deserts, caves, and human structures.",
        'wiki': 'https://en.wikipedia.org/wiki/Spider',
    },
    'squirrel': {
        'lifespan': '3-6 years in the wild (up to 15 years in captivity/zoos).',
        'habitat': "Native to most continents except Australia and Antarctica. They primarily inhabit arboreal environments in temperate and tropical forests, as well as urban parks.",
        'wiki': 'https://en.wikipedia.org/wiki/Squirrel',
    },
      'capybara': {
        'lifespan': '4-8 years in the wild (up to 12 years in captivity/zoos).',
        'habitat': "Native to South America. They thrive in lush, semi-aquatic habitats and are strictly tied to areas near water, such as swamps, marshes, rivers, and lakes.",
        'wiki': 'https://en.wikipedia.org/wiki/Capybara',
    },
    'frog': {
        'lifespan': '2-5 years in the wild (up to 15 years in captivity/zoos).',
        'habitat': "Distributed globally except for polar regions and the driest deserts. They are primarily found in moist, freshwater environments like ponds, swamps, and tropical rainforests.",
        'wiki': 'https://en.wikipedia.org/wiki/Frog',
    },
    'turtle': {
        'lifespan': '20-40 years in the wild (up to 80 years in captivity/zoos).',
        'habitat': "Distributed throughout most temperate and tropical regions worldwide. They occupy diverse environments ranging from oceans and freshwater rivers to arid deserts (tortoises).",
        'wiki': 'https://en.wikipedia.org/wiki/Turtle',
    },
    'rhino': {
        'lifespan': '35-45 years in the wild (up to 50 years in captivity/zoos).',
        'habitat': "Found in sub-Saharan Africa and parts of South and Southeast Asia. They thrive in tropical grasslands, savannas, shrublands, and dense swampy forests.",
        'wiki': 'https://en.wikipedia.org/wiki/Rhinoceros',
    },
    'tiger': {
        'lifespan': '10-15 years in the wild (up to 25 years in captivity/zoos).',
        'habitat': "Native to Asia, ranging from the freezing Russian Taiga to the tropical forests of India and Southeast Asia. They thrive in deep forests, mangrove swamps, and dense grasslands.",
        'wiki': 'https://en.wikipedia.org/wiki/Tiger',
    },
    'crocodile': {
        'lifespan': '30-50 years in the wild (up to 70 years in captivity/zoos).',
        'habitat': "Found in tropical regions of the Americas, Africa, Asia, and Australia. They inhabit freshwater rivers, lakes, wetlands, and sometimes brackish coastal mangrove swamps.",
        'wiki': 'https://en.wikipedia.org/wiki/Crocodile',
    },
    'wolf': {
        'lifespan': '6-8 years in the wild (up to 15 years in captivity/zoos).',
        'habitat': "Widely distributed across North America and Eurasia. They are highly adaptable, thriving in diverse biomes including tundra, temperate forests, mountains, and deserts.",
        'wiki': 'https://en.wikipedia.org/wiki/Wolf',
    },
    'horse': {
        'lifespan': '25-30 years in the wild (up to 35 years in captivity/zoos).',
        'habitat': "Originally native to the grasslands of Eurasia. Today, wild populations (Przewalski's) are restricted to Mongolia, while domesticated horses are found globally in pastures and savannas.",
        'wiki': 'https://en.wikipedia.org/wiki/Horse',
    },
    'elephant': {
        'lifespan': '60-70 years in the wild (JUST 50 years in captivity/zoos).',
        'habitat': "Found in sub-Saharan Africa and South/Southeast Asia. They adapt to diverse habitats including savannas, grasslands, dense tropical rainforests, and even deserts.",
        'wiki': 'https://en.wikipedia.org/wiki/Elephant',
    },
    'giraffe': {
        'lifespan': '10-15 years in the wild (up to 25 years in captivity/zoos).',
        'habitat': "Native to Africa, predominantly found in sub-Saharan regions. They thrive in arid and dry savannas, open grasslands, and woodlands rich in acacia trees.",
        'wiki': 'https://en.wikipedia.org/wiki/Giraffe',
    },
    'deer': {
        'lifespan': '10-20 years in the wild (up to 25 years in captivity/zoos).',
        'habitat': "Found on every continent except Antarctica and Australia (where they were introduced). They thrive in forests, woodlands, grasslands, wetlands, and mountain regions.",
        'wiki': 'https://en.wikipedia.org/wiki/Deer',
    },
    'goat': {
        'lifespan': '12-15 years in the wild (up to 18 years in captivity/zoos).',
        'habitat': "Wild goats are native to the mountains of Southwest Asia and Eastern Europe. Domesticated goats are found worldwide, thriving in farms, rocky hills, and dry scrublands.",
        'wiki': 'https://en.wikipedia.org/wiki/Goat',
    },
    'butterfly': {
        'lifespan': '2-4 weeks as an adult in the wild (some species, like the Monarch, up to 9 months).',
        'habitat': "Found on every continent except Antarctica. They live wherever flowering plants grow, including meadows, gardens, tropical rainforests, and mountain slopes.",
        'wiki': 'https://en.wikipedia.org/wiki/Butterfly',
    }

}
#==============================================================
#LẤY THÔNG TIN CỦA CON VẬT SAU KHI NHẬN DIỆN
#==============================================================
def format_info(animal_key):
    info = animal_info[animal_key]
    line1 = f"**Average Lifespan:** {info['lifespan']}"
    line2 = f"**Habitat & Distribution:** {info['habitat']}"
    line3 = f"[See more]({info['wiki']})"
    return line1 + "\n\n" + line2 + "\n\n" + line3

#==============================================================
# HÀM DỰ ĐOÁN
#==============================================================
def guessing(image):
    if image is None:
        error_msg = 'Please submit an image of one animal only'
        text_typing = ''
        for char in error_msg:          #HIỆU ỨNG GÕ CHỮ
            text_typing += char
            yield gr.update(value=text_typing, visible=True)
            time.sleep(0.01)
    else:
        selected_animal, confidence = predict_animal(image)

        # Ảnh không giống loài nào model biết -> báo thay vì đoán bừa
        if confidence < CONFIDENCE_THRESHOLD:
            msg = f"Hmm, I'm not confident this is one of the {len(CLASS_NAMES)} animals I know. Try a clearer photo!"
            typed = ''
            for char in msg:           #HIỆU ỨNG GÕ CHỮ
                typed += char
                yield gr.update(value=typed, visible=True)
                time.sleep(0.01)
            return

        title_line = f'This is a/an {selected_animal.upper()} ' #({confidence:.0%} sure)

        # HIỆU ỨNG GÕ CHỮ CHO TIÊU ĐỀ CỦA OUTPUT (KHÔNG NÊN XÓA)
        typed_title = ''
        for char in title_line:
            typed_title += char
            yield gr.update(value=f"## {typed_title}", visible=True)
            time.sleep(0.01)

        # HIỆU ỨNG GÕ CHỮ CHO NỘI DUNG CỦA OUTPUT (KHÔNG NÊN XÓA)
        info_full = format_info(selected_animal)
        typed_info = ''
        for char in info_full:
            typed_info += char
            yield gr.update(value=f"## {title_line}\n\n{typed_info}")
            time.sleep(0.005)

# =========================================================
# CSS - GIAO DIỆN NGƯỜI DÙNG BÊN DƯỚI
# =========================================================
custom_css = """
/* 1. Nền ứng dụng chính */
.gradio-container { /*Nền của web*/
    background-image: url("https://i.postimg.cc/RFMVV8ds/image.png") !important; /*Trích ảnh làm nền*/
    background-size: cover !important;  /*căng ảnh bằng với kích thước web*/
    background-position: center !important;   /*căng chỉnh vị trí ảnh ngay giữa*/
    background-repeat: no-repeat !important;    /*Ảnh sẽ không lặp lại khi màn hình người dùng quá to*/
    background-attachment: fixed !important;    /*Hình nền sẽ được cố định, không trôi theo tương tác ng dùng*/
}

/* 2. TITLE (Robot Zookeeper) */
/* 2.1 Ô */
#app-title {  /*Tiêu đề web*/
    width: fit-content !important;    /*Chiều rộng phụ thuộc vào độ dài nội dung*/
    max-width: 90% !important;    /*Độ rộng tối đa là 90% màn hình người dùng*/
    margin: 20px auto !important;   /*Canh lề tự động nằm giữa cửa sổ, 20px là chừa trống phía trên và dưới*/
    text-align: center !important;    /*Canh khung chữ nằm giữa hộp*/
    background-color: rgba(58, 84, 56, 0.55) !important;  /*Chỉnh màu của hộp tiêu đề*/
    backdrop-filter: blur(8px) !important;    /*Tạo độ mờ tự nhiên, tăng thẩm mỹ*/
    -webkit-backdrop-filter: blur(8px) !important;  /*Hỗ trợ mọi trình duyệt đều có thể hiển thị*/
    border-radius: 20px !important;   /*Bo góc hộp tiêu đề*/
    padding: 10px 32px !important;    /*Canh lề hộp tiêu đề so với cửa sổ trên&dưới là 10px và trái&&phải là 32px*/
}
/* 2.2 Chữ */
#app-title h1 { /*Thiết kế kiểu chữ cho tiêu đề*/
    font-size: 3.2rem !important;   /*Kích thước chữ thay đổi theo loại trình duyệt, thiết bị của người dùng (3.2rem)*/
    text-align: center !important;    /*Canh cho dòng chữ nằm giữa khung chữ*/
    margin: 0 !important;   /*Hộp chữ ôm sát chữ bên trong*/
}

/* 3. KHUNG UPLOAD ẢNH*/
#camera-wrapper,
#camera-wrapper > div { /*Chọn phần khung của hộp gr.Image và tất cả tập con*/
    position: relative !important;  /*Chuyển hệ tọa độ từ cửa sổ to bên ngoài vào bên trong khung nhỏ*/
    background: transparent !important;   /*Xóa nền mặc định của khung*/
    border: none !important;  /*Xóa viền*/
    padding: 0 !important;  /*Không canh lề, tất cả thành phần đều có kích thước bằng nhau*/
    /*box-shadow: none !important; Cài bóng cho khung ảnh-nếu thích*/
}

#zoo-image-input,
#zoo-image-input .image-container,
#zoo-image-input .upload-container,
#zoo-image-input [data-testid="image"],
#zoo-image-input .image-frame,
#zoo-image-input .selectable {    /*Dòng 1 là tập cha (lớp to nhất của gr.Image), kế tiếp là lớp hiển thị ảnh đã input, lớp dùng để click vào-bắt đầu tải ảnh lên, lớp hình ảnh hiển thị sau khi tải, lớp khung ảnh, cuối cùng là lớp giúp người dùng tương tác bằng chuột*/
    background-color: rgba(255, 255, 255, 0.1) !important;  /*Màu của hộp này là trắng, độ mờ 10% do có nhiều lớp chồng lên nhau nên sẽ mờ vừa đủ*/
    backdrop-filter: blur(4px) !important;  /* Dòng này và dòng dưới để tạo hiệu ứng mờ đẹp mắt, mọi trình duyệt đều hiển thị được*/
    -webkit-backdrop-filter: blur(4px) !important;
    border-radius: 15px !important;   /*Bo góc*/
    box-shadow: none !important;    /*Đổ bóng cho hộp nếu thích*/
}

#camera-wrapper::after {  /*Tạo 1 layer ko xác định nằm bên dưới layer camera-wrapper để chèn frame máy quay*/
    content: "" !important;   /*Layer nằm dưới phải có nội dung, ở đây để trống vì ko cần thiết*/
    position: absolute !important;    /*Đè lớp này lên trên khung gr.Image*/
    top: 0; left: 0; right: 0; bottom: 0; /*Lớp này căng đầy theo khung gr.Image*/
    background-image: url('https://i.postimg.cc/GmLTCPbH/image.png') !important;    /*Trích ảnh vào lớp này dưới dạng background*/
    background-size: 100% 100% !important;    /*Ảnh này lại căng đầy theo lớp vừa tạo*/
    background-position: center !important;  /*Ảnh nằm giữa lớp*/
    background-repeat: no-repeat !important;  /*Không lặp ảnh khi cửa sổ quá to so với ảnh*/
    pointer-events: none !important;    /*Vì là lớp nên có thể tương tác, chặn tương tác với các lớp nằm dưới với chuột nên tắt đi tính năng này*/
    z-index: 10 !important; /*Cho lớp này nằm lên trên cùng*/
}


/*10 dòng bên dưới giúp xóa tất cả các label của hộp gr.Image, gây mất thẩm mỹ*/
#zoo-image-input .block-label,
#zoo-image-input label,
#zoo-image-input span[data-testid="block-info"],
#zoo-image-input [class*="label"] {
    display: none !important;
    opacity: 0 !important;
    visibility: hidden !important;
    height: 0 !important;
    padding: 0 !important;
    margin: 0 !important;
}

/*4. NÚT SUBMIT*/
#submit-btn { /*Các cài đặt bên dưới áp dụng cho nút submit*/
    background-image: url('https://i.postimg.cc/MH5189fQ/image-removebg-preview.png') !important; /*Trích ảnh từ link*/
    background-size: contain !important;
    background-position: center !important;
    background-repeat: no-repeat !important;
    background-color: transparent !important;
    color: transparent !important;
    text-shadow: none !important;
    height: 100px !important;
    max-width: 60px !important;
    width: 60px !important;
    margin: 10px auto !important;
    border: none !important;
    box-shadow: none !important;
    border-radius: 35px !important;
    cursor: pointer;
    transition: transform 0.2s ease;
}
#submit-btn:hover { transform: scale(1.08) !important; }
#submit-btn:active { transform: scale(0.95) !important; }

/*5. NÚT CLEAR*/
#clear-btn {
    background-image: url('https://i.postimg.cc/cJxVKSqJ/img-icons8.png') !important;
    background-size: contain !important;
    background-position: center !important;
    background-repeat: no-repeat !important;
    background-color: transparent !important;
    color: transparent !important;
    text-shadow: none !important;
    border: none !important;
    box-shadow: none !important;
    max-width: 60px !important;
    width: 60px !important;
    height: 100px !important;
    padding: 0 !important;
    margin: 20px auto !important;
    border-radius: 35px !important;
    cursor: pointer;
    transition: transform 0.2s ease;
}
#clear-btn:hover { transform: scale(1.08) !important; }
#clear-btn:active { transform: scale(0.95) !important; }

/*4. Ô KẾT QUẢ GỘP CHUNG (result-box) - thay cho answer + info_box cũ */
#result-box {
    background-color: rgba(115, 115, 115, 0.9) !important;
    border-radius: 12px !important;   /* BO GÓC */
    padding: 16px 24px !important;    /* KHOẢNG TRỐNG bên trong */
    box-shadow: 0 4px 6px rgba(255,255,255,0.05);
    margin-top: 10px !important;
}

/* 4.1. DÒNG TIÊU ĐỀ "This is a ..." */

#result-box h2 {
    font-size: 2rem !important;
    text-align: center !important;
    margin: 0 0 10px 0 !important;
}

/* 4.2. HYPERLINK */
#result-box a {
    font-weight: bold !important;
    color: #2b6cb0 !important;
    text-decoration: underline !important;
}
"""
# ========================================================
# KHỞI CHẠY FRONTEND
# ========================================================

with gr.Blocks(css=custom_css, title='Animal Identifier') as demo:
    gr.Markdown("# 🖼️🦚Robot Zookeeper", elem_id="app-title")

  # BỌC KHUNG ẢNH VÀO TRONG MỘT BOX CÓ ID ĐỂ CSS PHỦ KHUNG CAMERA ĐÈ LÊN
    with gr.Group(elem_id="camera-wrapper"):
        image_input = gr.Image(
            type='numpy',
            height=630,
            elem_id="zoo-image-input"
        )
  # CÁC NÚT
    with gr.Row():
        submit_button = gr.Button(value="Let's go!", variant='primary', scale=1, elem_id="submit-btn")  #GẮN ID ĐỂ CSS NHẬN DIỆN
        clear_button = gr.ClearButton(value="Clear", components=[image_input], elem_id="clear-btn")   #GẮN ID ĐỂ CSS NHẬN DIỆN

  # Ô THÔNG TIN
    result_box = gr.Markdown(visible=False, elem_id="result-box")

  # ACTIVATIONS
    submit = submit_button.click(fn=guessing, inputs=image_input, outputs=result_box)
    clear = clear_button.click(fn=lambda: gr.update(value='', visible=False),
                       outputs=result_box, cancels=[submit])                #NGẮT,XÓA ẢNH ĐÃ TẢI LÊN, XÓA HỘP THOẠI OUTPUT

demo.launch(share=True)

Downloading...
From: https://drive.google.com/uc?id=1lD-76r1fRmsetxUGMF6KIK0kuO2wnjPX
To: /content/my_model.h5
100% 23.2M/23.2M [00:00<00:00, 45.3MB/s]


/tmp/ipykernel_997/1587457211.py:356: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, title='Animal Identifier') as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6eaca424db857ff0b6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import os
import tensorflow as tf

# Đảm bảo xóa bỏ cấu hình ép buộc Keras 2 cũ đi
if "TF_USE_LEGACY_KERAS" in os.environ:
    del os.environ["TF_USE_LEGACY_KERAS"]

# Định nghĩa lớp sửa lỗi BatchNormalization cho Keras 3
class CompatibleBatchNormalization(tf.keras.layers.BatchNormalization):
    def __init__(self, **kwargs):
        # Bỏ các tham số thừa của Keras 2 khiến Keras 3 bị lỗi
        kwargs.pop('renorm', None)
        kwargs.pop('renorm_clipping', None)
        kwargs.pop('renorm_momentum', None)
        super().__init__(**kwargs)

FILE_ID = "1tofYfOV-rLaMRC7LQzuhGfxfYTSf-USH"   # ID file của bạn
MODEL_PATH = "my_model.keras"                   # Tên file lưu mô hình (model mới lưu dạng .keras)

# Tải file mô hình (Nếu chưa có)
if not os.path.exists(MODEL_PATH):
    !pip install -q gdown
    !gdown {FILE_ID} -O {MODEL_PATH}

# LOAD MODEL: Sử dụng Keras 3 gốc kết hợp với lớp sửa lỗi tự chế ở trên
model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={'BatchNormalization': CompatibleBatchNormalization}
)

print("\n--- TẢI MÔ HÌNH THÀNH CÔNG RỒI BẠN NHÉ! ---")

# Tiếp tục các thư viện Gradio của bạn
import gradio as gr
import time   # delay type chữ
import random # testing


--- TẢI MÔ HÌNH THÀNH CÔNG RỒI BẠN NHÉ! ---
